# Complete Single-Cell RNA-seq Analysis Workflow in Python

This comprehensive workflow covers all essential steps for analyzing single-cell RNA sequencing data using Python, with a focus on the Scanpy ecosystem. This notebook is a runnable version of the original Markdown guide, adapted to use a sample dataset for immediate execution.

## Prerequisites and Setup

First, we install the necessary libraries. Running this in a notebook with `!` is convenient, but for a cleaner setup, it's recommended to run these commands in your terminal/command prompt within your project's virtual environment.

In [ ]:
# Essential libraries installation
%pip install scanpy pandas numpy matplotlib seaborn 'anndata>=0.8.0' 'scanpy>=1.9.0' scikit-learn umap-learn leidenalg scrublet gseapy session_info
# %pip install scvi-tools cellxgene-census cellxgene (optional, for advanced methods)

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
import warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3  # verbosity level: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Set random seed for reproducibility
np.random.seed(42)

ModuleNotFoundError: No module named 'scanpy'

## 1. Data Loading and Preprocessing

### 1.1 Loading Data

Since we don't have the original data files, we'll load a publicly available sample dataset from Scanpy: **3k PBMCs from 10x Genomics**. The code for loading other common formats is kept below for reference.

In [ ]:
# Method 1: Loading Scanpy's built-in PBMC dataset (for this runnable notebook)
adata = sc.datasets.pbmc3k()
adata.var_names_unique()  # Make variable names unique

# --- Reference Code for Other Data Formats ---

# Method 2: Loading 10x Genomics data (most common)
# For 10x HDF5 format
# adata = sc.read_10x_h5('filtered_feature_bc_matrix.h5')
# adata.var_names_unique()  # Make variable names unique
# adata.var['gene_symbols'] = adata.var_names # Store original gene symbols
# adata.var.set_index('gene_ids', inplace=True) # Use Ensembl IDs for var_names

# Method 3: Loading from 10x MTX format
# adata = sc.read_10x_mtx(
#     'filtered_feature_bc_matrix/',  # Path to the mtx folder
#     var_names='gene_symbols',       # use gene symbols for gene names (variables-axis index)
#     cache=True                      # write a cache file for faster subsequent reading
# )

# Method 4: Loading AnnData objects
# adata = sc.read_h5ad('data.h5ad')

# Method 5: Loading from CSV/TSV
# adata = sc.read_csv('expression_matrix.csv').T  # Transpose if genes are rows

# --- End of Reference Code ---

# Basic data inspection
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print(f"Data matrix shape: {adata.shape}")
print(f"Data type: {type(adata.X)}")

# Make data sparse if it isn't already (saves memory)
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

### 1.2 Initial Data Exploration

In [ ]:
# Basic statistics
print("First 5 cell barcodes:", adata.obs_names[:5])
print("First 5 gene names:", adata.var_names[:5])

# Check for mitochondrial and ribosomal genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # Human
# adata.var['mt'] = adata.var_names.str.startswith('mt-')  # Mouse
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

print(f"Number of mitochondrial genes: {adata.var['mt'].sum()}")
print(f"Number of ribosomal genes: {adata.var['ribo'].sum()}")

### 1.3 Quality Control Metrics

In [ ]:
# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True)

# The 'pct_counts_mt' and 'pct_counts_ribo' are now in adata.obs

# Additional useful metrics
adata.obs['log_total_counts'] = np.log10(adata.obs['total_counts'] + 1)
adata.obs['log_n_genes_by_counts'] = np.log10(adata.obs['n_genes_by_counts'] + 1)

### 1.4 Quality Control Visualization

In [ ]:
# Create QC plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(adata.obs['total_counts'], kde=True, ax=axes[0], bins=50)
axes[0].set_title('Total counts per cell')

sns.histplot(adata.obs['n_genes_by_counts'], kde=True, ax=axes[1], bins=50)
axes[1].set_title('Number of genes per cell')

sns.histplot(adata.obs['pct_counts_mt'], kde=True, ax=axes[2], bins=50)
axes[2].set_title('Mitochondrial gene %')

plt.tight_layout()
plt.show()

# Violin plots for a different view
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

# Scatter plots for relationships
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt')

### 1.5 Cell and Gene Filtering

Based on the plots above, we set thresholds to remove low-quality cells (dying cells, empty droplets) and potential doublets.

In [ ]:
# Save raw data before filtering
adata.raw = adata

# Print initial statistics
print(f"Initial: {adata.n_obs} cells, {adata.n_vars} genes")

# Filter cells based on QC metrics
# Typical thresholds for PBMC data (adjust based on your data):
min_genes = 200      # Minimum genes per cell
max_genes = 2500     # Maximum genes per cell (to filter potential doublets)
max_mt = 5           # Maximum mitochondrial percentage (low for healthy PBMCs)

# Apply filters
sc.pp.filter_cells(adata, min_genes=min_genes)
print(f"After min_genes filter: {adata.n_obs} cells")

adata = adata[adata.obs.n_genes_by_counts < max_genes, :]
print(f"After max_genes filter: {adata.n_obs} cells")

adata = adata[adata.obs.pct_counts_mt < max_mt, :]
print(f"After MT% filter: {adata.n_obs} cells")

# Filter genes (expressed in at least 3 cells)
sc.pp.filter_genes(adata, min_cells=3)
print(f"After gene filter: {adata.n_vars} genes")

print(f"\nFinal: {adata.n_obs} cells, {adata.n_vars} genes")

### 1.6 Doublet Detection (Optional but Recommended)

We use Scrublet to computationally identify cells that are likely doublets (two cells captured in the same droplet).

In [ ]:
try:
    import scrublet as scr
    
    # Use the raw, unfiltered data for doublet detection for better performance
    adata_scrub = adata.raw.to_adata().copy()
    sc.pp.normalize_total(adata_scrub, target_sum=1e4)
    sc.pp.log1p(adata_scrub)
    sc.pp.highly_variable_genes(adata_scrub)
    adata_scrub = adata_scrub[:, adata_scrub.var.highly_variable]
    
    # Initialize and run scrublet
    scrub = scr.Scrublet(adata_scrub.X, expected_doublet_rate=0.06)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                              min_cells=3, 
                                                              min_gene_variability_pctl=85, 
                                                              n_prin_comps=30)
    
    # Add results back to the original AnnData object (matching barcodes)
    adata.obs['doublet_score'] = doublet_scores
    adata.obs['predicted_doublet'] = predicted_doublets
    
    # Plot doublet score distribution
    scrub.plot_histogram()
    plt.show()
    
    # Filter doublets
    print(f"Detected {predicted_doublets.sum()} doublets")
    adata = adata[~adata.obs['predicted_doublet']].copy()
    print(f"After doublet removal: {adata.n_obs} cells")
    
except ImportError:
    print("Scrublet not installed. Skipping doublet detection.")
    print("Install with: pip install scrublet")

### 1.7 Normalization and Log Transformation

In [ ]:
# Normalize to 10,000 reads per cell (library size normalization)
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform (log(x + 1))
sc.pp.log1p(adata)

# Store the normalized data for later use (e.g., in visualization)
adata.layers['log1p_norm'] = adata.X.copy()

### 1.8 Feature Selection (Highly Variable Genes)

We focus our downstream analysis on the genes that show the most variation across cells, as these are most likely to be biologically meaningful.

In [ ]:
# Identify highly variable genes
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

# Plot highly variable genes
sc.pl.highly_variable_genes(adata)

# Print statistics
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

# Subset the data to only highly variable genes for downstream steps like PCA
# Note: We will use the full data for visualization later by using `adata.raw`
adata.raw = adata # Store the full (normalized, log-transformed) data
adata = adata[:, adata.var.highly_variable]

### 1.9 Data Integration (Example for Multiple Datasets)

This section is a placeholder showing how you would integrate multiple datasets to correct for batch effects. We are only using one dataset here, so this code is not run.

In [ ]:
# # Example for integrating multiple datasets
# # This section assumes you have multiple AnnData objects

# # Example with two datasets:
# adata1 = sc.read_10x_h5('dataset1.h5')
# adata2 = sc.read_10x_h5('dataset2.h5')

# # Perform QC and normalization on each dataset separately first!

# # Add batch information
# adata1.obs['batch'] = 'batch1'
# adata2.obs['batch'] = 'batch2'

# # Concatenate datasets
# adata_combined = sc.concat([adata1, adata2], axis=0, join='outer')

# # Find common highly variable genes
# sc.pp.highly_variable_genes(adata_combined, batch_key='batch')
# adata_combined = adata_combined[:, adata_combined.var.highly_variable]

# # For batch correction, you can use:
# # 1. Harmony (install: pip install harmonypy)
# # 2. Combat (built into scanpy)
# # 3. scVI (install: pip install scvi-tools)

# # Example with Combat:
# sc.pp.scale(adata_combined) # Scale first
# sc.pp.combat(adata_combined, key='batch')
# sc.tl.pca(adata_combined)
# sc.pp.neighbors(adata_combined)
# sc.tl.umap(adata_combined)
# sc.pl.umap(adata_combined, color='batch')

print("Data integration section is for demonstration and was not run.")

## 2. Dimensionality Reduction

We reduce the high-dimensional gene expression data to a lower-dimensional space to make patterns and structures more apparent.

### 2.1 Principal Component Analysis (PCA)

PCA is the first step. We regress out technical variables and scale the data before running PCA.

In [ ]:
# Regress out effects of total counts and mitochondrial percentage.
# Note: This is an alternative to simple scaling. Scaling is often sufficient.
# sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale data to unit variance and zero mean.
# Clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

# Compute PCA
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Plot the PCA elbow plot to determine the number of PCs to use
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

# Visualize PCA on the first two components
sc.pl.pca(adata, color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'])

### 2.2 Computing Neighborhood Graph

We build a graph where cells are nodes and edges connect cells with similar expression profiles. This is crucial for both clustering and UMAP visualization. We use the top PCs (e.g., 40, based on the elbow plot) for this.

In [ ]:
# Compute the neighborhood graph using the top 40 PCs
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

### 2.3 UMAP

UMAP (Uniform Manifold Approximation and Projection) is a non-linear dimensionality reduction technique that is excellent for visualizing the high-dimensional data in 2D or 3D.

In [ ]:
# Compute UMAP
sc.tl.umap(adata, random_state=42)

# Plot UMAP
sc.pl.umap(adata, color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'])

### 2.4 t-SNE (Alternative to UMAP)

t-SNE is another popular visualization method. UMAP is generally faster and better at preserving global structure.

In [ ]:
# Compute t-SNE (optional, UMAP is generally preferred)
sc.tl.tsne(adata, random_state=42, n_pcs=40)

# Plot t-SNE
sc.pl.tsne(adata, color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'])

## 3. Clustering

Now we group cells into clusters based on their expression similarity.

### 3.1 Leiden Clustering

The Leiden algorithm is a community detection algorithm applied to the neighborhood graph. We test several resolution parameters to find a suitable number of clusters.

In [ ]:
# Leiden clustering with different resolutions
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0]

for res in resolutions:
    sc.tl.leiden(adata, resolution=res, key_added=f'leiden_res_{res}')

# Visualize different resolutions
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
axes = axes.ravel()

for i, res in enumerate(resolutions):
    sc.pl.umap(adata, color=f'leiden_res_{res}', ax=axes[i], 
               title=f'Resolution {res}', frameon=False, show=False)

plt.tight_layout()
plt.show()

# Let's choose an optimal resolution for further analysis.
# 0.6 seems to provide a reasonable separation for PBMCs.
optimal_resolution = 0.6
adata.obs['clusters'] = adata.obs[f'leiden_res_{optimal_resolution}']

print(f"Number of clusters at resolution {optimal_resolution}: {len(adata.obs['clusters'].unique())}")

# Visualize the chosen clustering
sc.pl.umap(adata, color='clusters', legend_loc='on data', title=f'Leiden Clustering (res={optimal_resolution})')

### 3.2 Louvain Clustering (Alternative)

In [ ]:
# Louvain clustering
sc.tl.louvain(adata, resolution=0.6, key_added='louvain')

# Compare Leiden vs Louvain
sc.pl.umap(adata, color=['clusters', 'louvain'], ncols=2, title=['Leiden', 'Louvain'])

### 3.3 Clustering Quality Assessment

In [ ]:
# Silhouette score
from sklearn.metrics import silhouette_score

# Calculate silhouette score for different resolutions
silhouette_scores = []
for res in resolutions:
    cluster_labels = adata.obs[f'leiden_res_{res}'].astype(int)
    # Using a subset of cells for faster computation if needed
    score = silhouette_score(adata.obsm['X_pca'][:, :40], cluster_labels)
    silhouette_scores.append(score)
    print(f"Resolution {res}: Silhouette score = {score:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(resolutions, silhouette_scores, 'bo-')
plt.xlabel('Resolution')
plt.ylabel('Silhouette Score')
plt.title('Clustering Quality vs Resolution')
plt.grid(True)
plt.show()

## 4. Cell Type Annotation

The next crucial step is to assign biological cell identities to our computed clusters.

### 4.1 Marker Gene Analysis

We identify genes that are differentially expressed in each cluster compared to all other clusters. These are the 'marker genes'.

In [ ]:
# Find marker genes for each cluster using the Wilcoxon rank-sum test
sc.tl.rank_genes_groups(adata, 'clusters', method='wilcoxon', key_added='wilcoxon')

# Plot top marker genes as a heatmap
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, key='wilcoxon')

# Get marker genes as a DataFrame
marker_genes_df = sc.get.rank_genes_groups_df(adata, group=None, key='wilcoxon')
print(marker_genes_df.head(20))

# Save marker genes to a file
marker_genes_df.to_csv('marker_genes.csv', index=False)

### 4.2 Manual Annotation with Known Markers

We use a list of known marker genes for expected cell types in PBMCs to see which clusters express them.

In [ ]:
# Define known marker genes for different cell types in PBMCs
marker_dict = {
    'T-cells': ['CD3D', 'CD3E', 'CD3G'],
    'CD4+ T-cells': ['CD4', 'IL7R'],
    'CD8+ T-cells': ['CD8A', 'CD8B'],
    'B-cells': ['MS4A1', 'CD19', 'CD79A'],
    'NK cells': ['NKG7', 'GNLY', 'KLRD1'],
    'Monocytes': ['LYZ', 'CST3', 'CD14', 'FCGR3A'],
    'Dendritic Cells': ['FCER1A', 'CLEC10A'],
    'Platelets': ['PPBP']
}

# Plot expression of marker genes on UMAP
# We use use_raw=True to plot the full, un-subsetted data
for cell_type, genes in marker_dict.items():
    # Check which genes are present in the dataset
    present_genes = [g for g in genes if g in adata.raw.var_names]
    if present_genes:
        print(f"\n{cell_type} Markers: {present_genes}")
        sc.pl.umap(adata, color=present_genes, ncols=3, use_raw=True, title=cell_type)

### 4.3 Dot Plot for Marker Genes

A dot plot is an excellent way to summarize marker gene expression across all clusters. The size of the dot represents the percentage of cells in the cluster expressing the gene, and the color represents the average expression level.

In [ ]:
# Create a comprehensive marker gene list for the dot plot
all_markers = []
for genes in marker_dict.values():
    all_markers.extend(genes)

# Remove duplicates and check presence in the raw data
all_markers = list(set(all_markers))
present_markers = [g for g in all_markers if g in adata.raw.var_names]

# Create dot plot
sc.pl.dotplot(adata, present_markers, groupby='clusters', 
              use_raw=True, standard_scale='var')

### 4.4 Manual Cell Type Assignment

**This is a critical, and often subjective, step.** Based on the marker gene plots above, we can now assign identities to the clusters. This requires biological knowledge and careful interpretation.

**Note:** The mapping below is an **example** based on a typical run on the PBMC dataset. Your cluster numbers may differ! Always check your own dot plots and marker gene lists.

In [ ]:
# Based on the marker gene expression, manually assign cell types
cluster_to_celltype = {
    '0': 'CD4 T-cells',
    '1': 'CD14 Monocytes',
    '2': 'B-cells',
    '3': 'CD8 T-cells',
    '4': 'NK cells',
    '5': 'FCGR3A Monocytes',
    '6': 'Dendritic Cells',
    '7': 'Platelets'
    # Add more based on your clusters, or label as 'Unknown'
}

# Add cell type annotations to adata.obs
adata.obs['cell_type'] = adata.obs['clusters'].map(cluster_to_celltype).astype('category')

# Reorder categories for better plotting
if 'Unknown' not in adata.obs['cell_type'].cat.categories:
    adata.obs['cell_type'].cat.reorder_categories(
        list(cluster_to_celltype.values()), inplace=True
    )

# Visualize cell types on the UMAP
sc.pl.umap(adata, color='cell_type', legend_loc='on data', 
           legend_fontsize=8, legend_fontoutline=2, title="Cell Type Annotations")

### 4.5 Automated Annotation (Optional)

Tools like CellTypist can automate annotation using pre-trained models. This is a great way to get a first-pass annotation or validate manual findings.

In [ ]:
# Example with CellTypist (if installed: pip install celltypist)
try:
    import celltypist
    # It's recommended to run this on the un-normalized data
    adata_for_celltypist = adata.raw.to_adata()
    
    # Predict cell types
    predictions = celltypist.annotate(adata_for_celltypist, model='Immune_All_Low.pkl')
    
    # Add predicted labels to our main adata object
    adata.obs['predicted_celltype'] = predictions.predicted_labels['predicted_labels']
    
    # Compare manual vs. automated annotation
    sc.pl.umap(adata, color=['cell_type', 'predicted_celltype'], ncols=2)

except (ImportError, ModuleNotFoundError):
    print("CellTypist not installed or model not found. Skipping automated annotation.")
    print("Install with: pip install celltypist")

## 5. Differential Expression Analysis

Now that we have cell types, we can perform more targeted analyses.

### 5.1 Identify Marker Genes per Cell Type

This is similar to what we did for clusters, but now on our annotated cell types.

In [ ]:
# Find marker genes for each cell type
sc.tl.rank_genes_groups(adata, 'cell_type', method='wilcoxon', 
                        key_added='celltype_markers')

# Plot results as a heatmap
sc.pl.rank_genes_groups(adata, n_genes=5, key='celltype_markers')

# Get top markers for each cell type as a DataFrame
celltype_markers_df = sc.get.rank_genes_groups_df(adata, group=None, 
                                                  key='celltype_markers')
print(celltype_markers_df.head(20))

### 5.2 Compare Specific Cell Types

We can perform differential expression between specific groups of interest, for example, comparing CD4 T-cells to CD8 T-cells.

In [ ]:
# Example: Compare CD4 T-cells vs CD8 T-cells
sc.tl.rank_genes_groups(adata, 'cell_type', groups=['CD4 T-cells'], 
                        reference='CD8 T-cells', method='wilcoxon',
                        key_added='CD4_vs_CD8')

# Show results as a volcano plot
# Note: The default volcano plot in scanpy can be basic.
# For publication, you might export the data and use another library (e.g., gseapy, matplotlib).
sc.pl.volcano(adata, key='CD4_vs_CD8', x='logfoldchanges', y='pvals_adj')

# Show top differing genes in a violin plot
sc.pl.rank_genes_groups_violin(adata, groups=['CD4 T-cells'], 
                               key='CD4_vs_CD8', n_genes=10)

## 6. Visualization

Creating clear, publication-quality figures is a key outcome of the analysis.

### 6.1 Standard Plots

In [ ]:
# UMAP plots with different colorings
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Cell types
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Cell Types')

# Leiden clusters
sc.pl.umap(adata, color='clusters', ax=axes[1], show=False, title='Leiden Clusters')

# Example gene expression (T-cell marker)
if 'CD3D' in adata.raw.var_names:
    sc.pl.umap(adata, color='CD3D', ax=axes[2], show=False, use_raw=True, title='CD3D Expression')

plt.tight_layout()
plt.show()

### 6.2 Violin Plots

In [ ]:
# Violin plots for QC metrics by cell type
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             groupby='cell_type', rotation=90, multi_panel=True)

### 6.3 Heatmaps

In [ ]:
# Heatmap of top marker genes for each cell type
if 'celltype_markers_df' in locals():
    # Get top 5 genes per cell type
    top_genes = celltype_markers_df.groupby('group').head(5)['names'].tolist()
    top_genes = list(set(top_genes)) # a gene can be a marker for multiple types
    
    sc.pl.heatmap(adata, top_genes, groupby='cell_type', 
                  use_raw=True, show_gene_labels=True, standard_scale='var')

### 6.4 Dot Plots

In [ ]:
# A more focused dot plot for key markers
if 'present_markers' in locals():
    sc.pl.dotplot(adata, present_markers, groupby='cell_type', 
                  use_raw=True, standard_scale='var')

### 6.5 Advanced Visualization with Marsilea

Marsilea allows for creating publication-quality composable visualizations, such as complex heatmaps and dot plots with flexible annotations.

In [ ]:
# Install marsilea if needed
try:
    import marsilea as ma
    import marsilea.plotter as mp
except ImportError:
    %pip install marsilea
    import marsilea as ma
    import marsilea.plotter as mp
import matplotlib.pyplot as plt

In [ ]:
# Prepare data for Marsilea plotting
# We need to organize markers by cell type for better visualization
genes_category = []
genes_list = []
for ct, gs in marker_dict.items():
    for g in gs:
        if g in adata.raw.var_names:
            genes_list.append(g)
            genes_category.append(ct)

# Aggregate expression by cell type
# Ensure we use the raw data to include all markers
adata_raw = adata.raw.to_adata()
agg = sc.get.aggregate(adata_raw[:, genes_list], by='cell_type', func=['mean', 'count_nonzero'])
agg.obs['cell_counts'] = adata.obs['cell_type'].value_counts()

# Extract data for plotting
# Convert to numpy arrays to ensure compatibility
agg_exp = np.array(agg.layers['mean'])
agg_count = np.array(agg.layers['count_nonzero'])
agg_cell_counts = agg.obs['cell_counts'].to_numpy()

# Calculate fraction of cells expressing each gene
agg_fraction = agg_count / agg_cell_counts[:, np.newaxis]

In [ ]:
# Matrix Plot
h, w = agg_exp.shape

# Create Heatmap
m = ma.Heatmap(
    agg_exp,
    height=h * 0.4,
    width=w * 0.3,
    cmap="Blues",
    linewidth=0.5,
    linecolor="lightgray",
    label="Mean Expression"
)

# Group columns by marker category (cell type)
m.group_cols(genes_category, order=list(marker_dict.keys()))

# Add annotations
m.add_top(mp.Labels(genes_list), pad=0.1)
m.add_top(mp.Chunk(list(marker_dict.keys()), rotation=90), pad=0.1)
m.add_right(mp.Labels(agg.obs_names, align="center"), pad=0.1)
m.add_left(mp.Numbers(agg_cell_counts, color="#EEB76B", label="Count"), pad=0.1)

# Render
m.add_legends()
m.render()

In [ ]:
# Dot Plot
m = ma.SizedHeatmap(
    size=agg_fraction,
    color=agg_exp,
    cluster_data=agg_fraction,
    height=h * 0.4,
    width=w * 0.3,
    edgecolor="lightgray",
    cmap="Reds",
    size_legend_kws=dict(
        colors="gray",
        title="Fraction of cells (%)",
        labels=["20%", "40%", "60%", "80%", "100%"],
        show_at=[0.2, 0.4, 0.6, 0.8, 1.0],
    ),
    color_legend_kws=dict(title="Mean expression"),
)

m.group_cols(genes_category, order=list(marker_dict.keys()))
m.add_top(mp.Labels(genes_list), pad=0.1)
m.add_top(mp.Chunk(list(marker_dict.keys()), rotation=90), pad=0.1)
m.add_right(mp.Labels(agg.obs_names, align="center"), pad=0.1)
m.add_dendrogram("left", pad=0.1)

m.add_legends()
m.render()

### 6.6 Interactive Visualization

For interactive exploration, `cellxgene` is an excellent tool. You can launch it from the command line after saving your `AnnData` object.

In [ ]:
# Save data for interactive visualization with cellxgene
# First install: pip install cellxgene
adata.write('analyzed_pbmc_data.h5ad')

# You can then launch cellxgene from your terminal:
# cellxgene launch analyzed_pbmc_data.h5ad --open

print("Data saved as 'analyzed_pbmc_data.h5ad'")
print("To explore interactively, run this in your terminal:")
print("cellxgene launch analyzed_pbmc_data.h5ad --open")

## 7. Downstream Analysis

Here are examples of more advanced analyses.

### 7.1 Trajectory Inference

PAGA (Partition-based graph abstraction) can infer trajectories and pseudotime, for example, to model differentiation processes. Here we use it to visualize the relationships between our identified cell types.

In [ ]:
# Compute PAGA to see relationships between cell types
sc.tl.paga(adata, groups='cell_type')

# Plot PAGA graph, showing connection strength between clusters
sc.pl.paga(adata, color='cell_type', layout='fr', threshold=0.03, 
           node_size_scale=2, node_size_power=0.5, edge_width_scale=0.5)

# We can also use PAGA to initialize the UMAP for a trajectory-aware layout
sc.tl.umap(adata, init_pos='paga', random_state=42)
sc.pl.umap(adata, color=['cell_type', 'total_counts'], title=['Cell Types (PAGA-initialized UMAP)', 'Total Counts'])

### 7.2 Gene Set Enrichment Analysis

We can check if the marker genes for a specific cell type are enriched in certain biological pathways (e.g., Gene Ontology terms).

In [ ]:
try:
    import gseapy as gp
    
    # Get marker genes for a specific cell type, e.g., NK cells
    nk_cell_markers = celltype_markers_df[celltype_markers_df['group'] == 'NK cells']
    
    # Get top 100 upregulated genes
    top_genes = nk_cell_markers.sort_values('logfoldchanges', ascending=False).head(100)['names'].tolist()
    
    # Run enrichment analysis using Enrichr
    if top_genes:
        enr = gp.enrichr(gene_list=top_genes,
                         gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human'],
                         organism='Human',
                         outdir=None)
        
        # Display top results
        print("\n--- GSEA Results for NK cells ---")
        print(enr.results.head(10))
        
        # Plot results
        gp.barplot(enr.results, title='Enrichment for NK Cell Markers')
    else:
        print("No marker genes found for NK cells to perform GSEA.")
    
except ImportError:
    print("gseapy not installed. Skipping GSEA. Install with: pip install gseapy")

### 7.3 Cell-Cell Communication Analysis

This type of analysis requires specialized tools to infer signaling interactions between cell types based on ligand-receptor expression. Tools like CellPhoneDB or LIANA are commonly used.

In [ ]:
# Placeholder for cell communication analysis
print("Cell-cell communication analysis typically requires specialized tools.")
print("Popular options include:")
print("1. LIANA+ (Python): pip install liana-py")
print("2. CellPhoneDB (Python wrapper): pip install cellphonedb")
print("3. CellChat (R package)")

# Example pseudocode using LIANA+:
# import liana
# liana.mt.rank_aggregate(
#     adata,
#     groupby='cell_type',
#     resource_name='cellphonedb',
#     use_raw=False, # Use processed, scaled data
#     verbose=True
# )
# liana.pl.dotplot(adata, n_fn=20) # 'fn' stands for factor number

## 8. Saving Results and Reproducibility

### 8.1 Save Analysis Results

We save the final `AnnData` object, which contains all data, metadata, and analysis results, as well as key tables as CSV files.

In [ ]:
# Save the complete analysis object
adata.write('final_analysis.h5ad')

# Save specific results as CSV files
adata.obs[['cell_type', 'clusters', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt']].to_csv('cell_annotations_and_qc.csv')

if 'celltype_markers_df' in locals():
    celltype_markers_df.to_csv('celltype_markers.csv', index=False)

print("Analysis complete! Files saved:")
print("- final_analysis.h5ad: Complete AnnData object")
print("- cell_annotations_and_qc.csv: Cell annotations and QC metrics")
print("- celltype_markers.csv: Marker genes for each cell type")
print("- marker_genes.csv: Marker genes for each cluster")

### 8.2 Generate Analysis Report

A simple text report summarizing the key findings and parameters.

In [ ]:
# Create a summary report
report = f"""
# Single-Cell RNA-seq Analysis Report

## Dataset Summary
- **Total cells analyzed**: {adata.n_obs:,}
- **Total genes**: {adata.raw.n_vars:,} (before filtering)
- **Number of cell types identified**: {len(adata.obs['cell_type'].unique())}
- **Number of clusters**: {len(adata.obs['clusters'].unique())}

## Cell Type Distribution
{adata.obs['cell_type'].value_counts().to_string()}

## Quality Control Summary (Post-filtering)
- **Mean genes per cell**: {adata.obs['n_genes_by_counts'].mean():.0f}
- **Mean UMI per cell**: {adata.obs['total_counts'].mean():.0f}
- **Mean mitochondrial %**: {adata.obs['pct_counts_mt'].mean():.2f}%

## Analysis Parameters
- **Normalization**: Library size normalized to 10,000, log-transformed
- **Feature selection**: {adata.n_vars} highly variable genes used for PCA
- **Dimensionality reduction**: PCA (40 components), UMAP
- **Clustering**: Leiden algorithm (resolution = {optimal_resolution})
- **Differential expression**: Wilcoxon rank-sum test
"""

# Save report to a markdown file
with open('analysis_report.md', 'w') as f:
    f.write(report)

print(report)
print("\nAnalysis report saved as 'analysis_report.md'")

## 9. Best Practices and Troubleshooting

### 9.1 Best Practices for Reproducibility

In [ ]:
# Set seeds for reproducibility (already done at the start)
import random
random.seed(42)
np.random.seed(42)

# Log analysis parameters in a structured way
analysis_params = {
    'min_genes_per_cell': min_genes,
    'max_genes_per_cell': max_genes,
    'max_mt_percent': max_mt,
    'normalization_target': 1e4,
    'n_highly_variable_genes': int(adata.var['highly_variable'].sum()),
    'leiden_resolution': optimal_resolution,
    'n_neighbors': 15,
    'n_pcs': 40
}

# Save parameters to a JSON file
import json
with open('analysis_parameters.json', 'w') as f:
    json.dump(analysis_params, f, indent=4)

print("Analysis parameters saved to 'analysis_parameters.json'")

# Save session info to see package versions
import session_info
session_info.show(html=False, dependencies=True)

### 9.2 List of Generated Files

Here are all the files created during this workflow.

In [ ]:
!ls -lh

## 10. Next Steps

This notebook provides a complete, foundational workflow. From here, you can:

1.  **Refine Analysis:** Adjust filtering, clustering, and annotation parameters based on a deeper look at the data and your biological question.
2.  **Advanced Topics:** Explore trajectory inference, cell-cell communication, or integration with other data modalities (e.g., spatial, ATAC-seq).
3.  **Biological Interpretation:** Dive deep into the differential expression results, perform GSEA on more comparisons, and formulate hypotheses for validation.
4.  **Publication:** Generate high-resolution, publication-quality figures by customizing Matplotlib/Seaborn settings or using specialized plotting libraries.

Happy analyzing! 🧬📊